# Supply-chain demand planning — multi-agent on Oracle AI Database

You're going to build a planner-assistant out of three agents — a
**supervisor** that decomposes the request and synthesises the answer,
plus two specialists (`demand_analyst` and `policy_agent`) — each backed
by a different Oracle persistence layer. Every embedding is computed
**in-database** via an ONNX model loaded into Oracle. No external
embedding API is called.

![Demand planning with a human in the loop](../images/oracle_multi_agent_demand.png)

![Multi-agent demand planning on Oracle AI Database](../images/multi_agent_overview.png)

The Codespace has already done the heavy lifting:

- The `AGENT` Oracle user exists with a vector memory pool.
- The `ALL_MINILM_L12_V2` ONNX model is loaded inside Oracle (384-dim).
- The [`harisss/Supplychain`](https://huggingface.co/datasets/harisss/Supplychain)
  Hugging Face dataset has been aggregated into the top-12 demand reports +
  a buy-volume policy memo, written into `OracleVS`.
- Two planner preferences (Priya = conservative, Michael = aggressive)
  are pre-seeded in `AsyncOracleStore`.

Your job is to **wire the agents that consume all of this** and get the
supervisor to produce a Priya-aware buy recommendation.

## At a glance — 9 TODOs across 12 parts

| Part | Topic | TODO |
|---|---|---|
| 1 | Setup & connectivity | — |
| 2 | In-DB embeddings (`OracleEmbeddings`, `ALL_MINILM_L12_V2`) | **TODO 1** |
| 3 | `OracleVS` — the vector knowledge base | **TODO 2** |
| 4 | `AsyncOracleStore` — long-term cross-thread memory | **TODO 3** |
| 5 | `AsyncOracleSaver` — per-thread checkpoints | **TODO 4** |
| 6 | `OracleSemanticCache` — sanity-check with two timed calls | **TODO 5** |
| 7 | Naive substring vs semantic vector search | **TODO 6** |
| 8 | `demand_analyst` specialist + its tool | **TODO 7** |
| 9 | `policy_agent` specialist + its two tools | **TODO 8** |
| 10 | Supervisor + end-to-end run | **TODO 9** |
| 11 | `OracleChatMessageHistory` — standalone primitive | — |
| 12 | Cleanup | — |

Each TODO is followed by a **hard-stop checkpoint** cell that asserts the
TODO is correct. If a checkpoint fails, fix the TODO above it before
moving on.

📖 Per-part guides live in `docs/`. Each TODO points at the relevant guide.


# Part 1 — Setup & connectivity

Pick the chat-model provider (OpenAI or OCI), validate the keys, and open
the three Oracle connections we'll need.

📖 [`docs/part-1-setup.md`](../docs/part-1-setup.md)

![Reference architecture by provider layer](../images/provider_layers.png)


## 1.1 Pick the LLM provider and validate credentials

The chat model is provider-aware via `LLM_PROVIDER`. Both endpoints speak
the OpenAI wire protocol, so the same `ChatOpenAI` client works in both
modes — only the `base_url` and `api_key` change.

| `LLM_PROVIDER` | Required env vars |
|---|---|
| `openai` (default) | `OPENAI_API_KEY`, optional `LLM_MODEL` (default `gpt-5.5`) |
| `oci` | `OCI_GENAI_API_KEY`, `OCI_GENAI_ENDPOINT`, `LLM_MODEL` (default `xai.grok-4-1-fast-reasoning`) |

Embeddings always use the in-DB ONNX model (Part 2), so no external
embedding key is ever required.

In [ ]:
import getpass
import os

LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "openai").lower()
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not os.environ.get("OCI_GENAI_API_KEY"):
        os.environ["OCI_GENAI_API_KEY"] = getpass.getpass("OCI GenAI API key: ")
    os.environ.setdefault("OCI_GENAI_ENDPOINT",
        "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com")
    os.environ.setdefault("LLM_MODEL", "xai.grok-4-1-fast-reasoning")
else:
    assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")

LLM_MODEL = os.environ["LLM_MODEL"]


def chat_model_kwargs(**extra):
    """Build kwargs for `ChatOpenAI` that respect `LLM_PROVIDER`."""
    kwargs = dict(extra)
    if LLM_PROVIDER == "oci":
        kwargs.setdefault("base_url", os.environ["OCI_GENAI_ENDPOINT"])
        kwargs.setdefault("api_key", os.environ["OCI_GENAI_API_KEY"])
    return kwargs


ORACLE_USER = os.environ.get("ORACLE_USER", "AGENT")
ORACLE_PASSWORD = os.environ.get("ORACLE_PASSWORD", "AgentPwd_2025")
ORACLE_DSN = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")

ONNX_MODEL = os.environ.get("ONNX_EMBED_MODEL", "ALL_MINILM_L12_V2")
ONNX_DIMS = int(os.environ.get("ONNX_EMBED_DIM", "384"))

print(f"Chat LLM:   {LLM_PROVIDER}/{LLM_MODEL}")
print(f"Embeddings: in-DB ONNX / {ONNX_MODEL} ({ONNX_DIMS} dims)")
print(f"Oracle:     {ORACLE_USER}@{ORACLE_DSN}")

## 1.2 Open the three Oracle connections

`oracledb` keeps sync and async APIs separate. We open:

- **`oracle_client`** (sync) — used by `OracleVS`, `OracleSemanticCache`,
  and `OracleChatMessageHistory`.
- **`saver_conn`** (async) — used by `AsyncOracleSaver`.
- **`store_conn`** (async) — used by `AsyncOracleStore`.

All three connect to the same database; from Oracle's perspective this is
one workload in one schema.

In [ ]:
import oracledb

connect_kwargs = {"user": ORACLE_USER, "password": ORACLE_PASSWORD, "dsn": ORACLE_DSN}
if ORACLE_USER.lower() == "sys":
    connect_kwargs["mode"] = oracledb.AUTH_MODE_SYSDBA

oracle_client = oracledb.connect(**connect_kwargs)             # sync
saver_conn = await oracledb.connect_async(**connect_kwargs)    # async — saver
store_conn = await oracledb.connect_async(**connect_kwargs)    # async — store

print("✅ three oracledb connections open.")

> **Key takeaway — Part 1.** Provider-aware `ChatOpenAI` (one helper
> function does the OCI vs OpenAI swap), three Oracle connections for the
> three persistence primitives.

# Part 2 — In-DB embeddings with `OracleEmbeddings`

The Codespace pre-loaded an ONNX model named **`ALL_MINILM_L12_V2`** into
your Oracle schema. It produces **384-dim** float vectors and is invoked
from inside the database via `VECTOR_EMBEDDING(...)` — no Python
round-trip to a remote embedding API.

`langchain_oracledb.OracleEmbeddings` is a LangChain `Embeddings`
implementation that wraps that SQL call, so you can pass it anywhere a
LangChain vector store or cache expects an embedder.

📖 [`docs/part-2-embeddings.md`](../docs/part-2-embeddings.md)

## 2.1 Wire up `OracleEmbeddings`

> **TODO 1.** Construct an `OracleEmbeddings` so the rest of the notebook
> can embed text in-database (no external API calls).
>
> **Required kwargs**
> - `conn` = `oracle_client` (the sync handle from Part 1)
> - `params` = `{"provider": "database", "model": ONNX_MODEL}`
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-2-embeddings.md](../docs/part-2-embeddings.md)

In [ ]:
from langchain_oracledb import OracleEmbeddings

embeddings = OracleEmbeddings(
    conn=oracle_client,
    params={"provider": "database", "model": ONNX_MODEL},
)

In [ ]:
# Hard-stop checkpoint: TODO 1 must work before later parts run.
assert embeddings is not None, "❌ TODO 1: `embeddings` is still None — build the OracleEmbeddings."
_vec = embeddings.embed_query("smoke test for the in-DB embedder")
assert isinstance(_vec, list), "❌ TODO 1: embed_query must return a list of floats."
assert len(_vec) == ONNX_DIMS, (
    f"❌ TODO 1: embedding length {len(_vec)} != {ONNX_DIMS}. "
    f"Did you point OracleEmbeddings at the wrong model?"
)
print(f"✅ TODO 1 passed — embed_query returned a {len(_vec)}-dim vector from in-DB ONNX.")

> **Key takeaway — Part 2.** The embedder is now a normal LangChain
> object, but every `embed_query`/`embed_documents` call resolves
> server-side. No external embedding API, no network hop, no per-token
> billing for embeddings.

# Part 3 — `OracleVS` — the vector knowledge base

`OracleVS` is the canonical LangChain vector store for Oracle. It's
already pre-seeded by `app/scripts/seed_supplychain.py` with:

- 12 **demand reports** (`{"type": "demand_report", "product": <name>}`),
- 1 **policy memo** (`{"type": "policy", "name": "planner_buy_volume"}`).

You'll **re-instantiate** it here so the agents can use it later.
Re-instantiating against the same `table_name` reuses the existing
table; no data is dropped.

📖 [`docs/part-3-vector-store.md`](../docs/part-3-vector-store.md)

## 3.1 Instantiate `OracleVS`

> **TODO 2.** Get a Python handle on the demand-report vector store that
> was populated by `app/scripts/seed_supplychain.py` during the Codespace
> setup. Re-instantiating against an existing table is non-destructive —
> no data is dropped.
>
> **Required kwargs**
> - `client` = `oracle_client`
> - `embedding_function` = `embeddings` (from TODO 1)
> - `table_name` = `"supplychain_demand"`
> - `distance_strategy` = `DistanceStrategy.COSINE`
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-3-vector-store.md](../docs/part-3-vector-store.md)

In [ ]:
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb import OracleVS

oracle_vs = OracleVS(
    client=oracle_client,
    embedding_function=embeddings,
    table_name="supplychain_demand",
    distance_strategy=DistanceStrategy.COSINE,
)

In [ ]:
# Hard-stop checkpoint: TODO 2 — the pre-seeded data should be visible.
assert oracle_vs is not None, "❌ TODO 2: `oracle_vs` is still None."
_hits = oracle_vs.similarity_search("planner buy volume policy", k=1)
assert _hits, "❌ TODO 2: similarity_search returned no rows — is the table populated?"
assert any("policy" in (h.metadata or {}).get("type", "") for h in _hits), (
    "❌ TODO 2: a query for the policy didn't surface the policy memo. "
    "Either the seed step didn't run, or you're pointing at the wrong table."
)
print(f"✅ TODO 2 passed — OracleVS handle live; pre-seeded policy memo retrievable.")

> **Key takeaway — Part 3.** Re-instantiating an `OracleVS` against an
> existing table is non-destructive: you get a Python handle to query
> and the agents can `.similarity_search(...)` it.

# Part 4 — `AsyncOracleStore` — long-term cross-thread memory

`AsyncOracleStore` implements LangGraph's `BaseStore` interface — a
hierarchical key-value store with an optional vector index. Data here
**survives across `thread_id`s**, which is why we use it for things like
"this planner's saved preferences."

Already pre-seeded for you:

- `("users", "priya",   "memories") / "pref-conservative"` → *"Priya consistently prefers conservative buy volumes …"*
- `("users", "michael", "memories") / "pref-aggressive"`   → *"Michael chases category leaders …"*

📖 [`docs/part-4-store.md`](../docs/part-4-store.md)

![Unified memory core for AI agents](../images/agent_memory.png)

> Memory splits first by **duration** (short-term vs long-term vs
> coordination), then by **cognitive function** (working, episodic,
> procedural, semantic, persona). `AsyncOracleStore` covers the
> long-term, cross-thread tier on the right of the diagram —
> facts that outlive any single conversation.


## 4.1 Instantiate the store with an HNSW vector index

> **TODO 3.** Build a LangGraph `Store` backed by Oracle. Memories under
> `("users", user_id, "memories")` will survive across `thread_id`s.
>
> **Required args**
> - first positional arg: `store_conn` (the async handle from Part 1)
> - `index` = an HNSW config (see below)
> - `table_suffix` = `"agent_memory"` (must match the seed script)
>
> **HNSW index config**
> ```python
> {
>     "dims":  ONNX_DIMS,
>     "embed": embeddings,
>     "fields": ["note"],
>     "index_type": {"type": "hnsw", "distance_metric": "COSINE"},
> }
> ```
>
> Then call `await agent_store.setup()` — idempotent.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-4-store.md](../docs/part-4-store.md)

In [ ]:
from langgraph_oracledb.store.oracle import AsyncOracleStore

agent_store = AsyncOracleStore(
    store_conn,
    index={
        "dims": ONNX_DIMS,
        "embed": embeddings,
        "fields": ["note"],
        "index_type": {"type": "hnsw", "distance_metric": "COSINE"},
    },
    table_suffix="agent_memory",
)
await agent_store.setup()

In [ ]:
# Hard-stop checkpoint: TODO 3 — Priya's preference must be retrievable.
assert agent_store is not None, "❌ TODO 3: `agent_store` is still None."
_priya = await agent_store.aget(("users", "priya", "memories"), "pref-conservative")
assert _priya is not None, (
    "❌ TODO 3: Priya's seeded memory wasn't found. Either your `table_suffix` "
    "doesn't match the seed script's, or the seed step failed."
)
assert "conservative" in (_priya.value or {}).get("note", "").lower(), (
    "❌ TODO 3: the retrieved memory doesn't mention 'conservative'. "
    "Re-run app/scripts/seed_supplychain.py."
)
print("✅ TODO 3 passed — AsyncOracleStore live; pre-seeded user memory retrievable.")

> **Key takeaway — Part 4.** The store is hierarchical (`("users",
> user_id, "memories")`) and survives across `thread_id`s — the right
> place for *who* the planner is and *what they prefer*.

# Part 5 — `AsyncOracleSaver` — per-thread checkpoints

`AsyncOracleSaver` is LangGraph's checkpointer. It snapshots every step
of the agent graph keyed by `thread_id`. This is **short-term memory**,
scoped to one conversation. Distinct from `AsyncOracleStore`, which
holds facts that outlive any single conversation.

📖 [`docs/part-5-saver.md`](../docs/part-5-saver.md)

## 5.1 Instantiate + setup the saver

> **TODO 4.** Build a LangGraph checkpointer that snapshots every step of
> the agent graph, keyed by `thread_id`. No index config — just the async
> connection.
>
> **Required arg**
> - first positional arg: `saver_conn` (the async handle from Part 1)
>
> Then call `await saver.setup()` — also idempotent.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-5-saver.md](../docs/part-5-saver.md)

In [ ]:
from langgraph_oracledb.checkpoint.oracle import AsyncOracleSaver

saver = AsyncOracleSaver(saver_conn)
await saver.setup()

In [ ]:
# Hard-stop checkpoint: TODO 4 — the saver should report a thread list,
# even if empty.
assert saver is not None, "❌ TODO 4: `saver` is still None."
_threads = []
async for t in saver.alist(None):
    _threads.append(t)
    if len(_threads) >= 1:
        break
print(f"✅ TODO 4 passed — AsyncOracleSaver live (existing threads visible: {len(_threads)}+).")

> **Key takeaway — Part 5.** Saver = short-term, per-thread. Store =
> long-term, cross-thread. The agent's
> `.compile(checkpointer=..., store=...)` in Part 10 is where both
> memory horizons converge.

# Part 6 — `OracleSemanticCache` — caching LLM calls

A semantic cache stores `(prompt, response)` pairs keyed by **vector
similarity of the prompt**, so even paraphrased prompts can hit. We'll
configure one against `oracle_client`, then run a two-call timed demo on
a plain `ChatOpenAI`.

📖 [`docs/part-6-cache.md`](../docs/part-6-cache.md)

![Semantic cache — every prompt looked up in Oracle before the LLM](../images/semantic_cache.png)


## 6.1 Configure the cache

In [ ]:
from langchain_oracledb.cache import OracleSemanticCache

semantic_cache = OracleSemanticCache(
    client=oracle_client,
    embedding=embeddings,
    table_name="langchain_demand_cache",
    distance_strategy="COSINE",
    score_threshold=0.05,
)
print("OracleSemanticCache configured.")

## 6.2 Sanity-check the cache with two timed calls

> **TODO 5.** Prove the cache lookup beats a fresh LLM round-trip.
>
> **Steps**
> 1. Build `cache_demo_model = ChatOpenAI(model=LLM_MODEL, cache=semantic_cache, max_tokens=120, **chat_model_kwargs())`.
> 2. `invoke()` once with `PROMPT`, time it → `r1`, `miss`.
> 3. `invoke()` again with the **same** prompt, time it → `r2`, `hit`.
> 4. Print both elapsed times and the speedup.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-6-cache.md](../docs/part-6-cache.md)

In [ ]:
import time
from langchain_openai import ChatOpenAI

cache_demo_model = ChatOpenAI(
    model=LLM_MODEL,
    cache=semantic_cache,
    max_tokens=120,
    **chat_model_kwargs(),
)

PROMPT = "In one sentence: what is supply-chain demand planning?"

t0 = time.perf_counter()
r1 = cache_demo_model.invoke(PROMPT)
miss = time.perf_counter() - t0

t0 = time.perf_counter()
r2 = cache_demo_model.invoke(PROMPT)
hit = time.perf_counter() - t0

print(f"first call  (miss): {miss:.2f}s")
print(f"second call ( hit): {hit:.2f}s")
print(f"speedup:           {miss / max(hit, 0.001):.1f}x")
print()
print(r2.content)

In [ ]:
# Hard-stop checkpoint: TODO 5 — both calls must succeed and return the
# same response.
assert r1 is not None and r2 is not None, "❌ TODO 5: r1 or r2 wasn't bound — did you run both invocations?"
assert r1.content == r2.content, (
    "❌ TODO 5: the two invocations returned different responses. "
    "Either the cache isn't connected, or score_threshold is too tight."
)
print("✅ TODO 5 passed — cache returns the stored response on the second call.")

> **Key takeaway — Part 6.** The cache lives in the same Oracle DB as
> everything else, uses the same in-DB embedder, and is opt-in per chat
> model.

# Part 7 — Naive substring vs semantic vector search

The planner asks: *"How are our soccer-related items moving?"* But the
catalogue uses brand language (*FIFA*, *Brazuca*, *match ball*, *cleat*).
The literal word **soccer** never appears in any of the seeded reports.

This is the aha moment: keyword search returns 0; vector search returns
the actually-relevant SKUs.

📖 [`docs/part-7-search-comparison.md`](../docs/part-7-search-comparison.md)

## 7.1 Show naive vs semantic side by side

> **TODO 6.** Prove that the literal word *"soccer"* never appears in any
> seeded demand report, but a vector search for the same concept finds the
> relevant football SKUs.
>
> **Steps**
> 1. **Naive side.** Pull every demand-report row and regex-search each
>    `.page_content` for `\bsoccer\b`. Bind matching product names to `naive`.
>    Useful starter:
>    ```python
>    all_hits = oracle_vs.similarity_search("demand report", k=50)
>    all_reports = [h for h in all_hits if (h.metadata or {}).get("type") == "demand_report"]
>    ```
> 2. **Semantic side.** `oracle_vs.similarity_search("soccer merchandise demand", k=5)`. Bind to `hits`.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-7-search-comparison.md](../docs/part-7-search-comparison.md)

In [ ]:
import re

# Naive substring filter — fetch all demand_report rows and grep them.
all_hits = oracle_vs.similarity_search("demand report", k=50)
all_reports = [h for h in all_hits if (h.metadata or {}).get("type") == "demand_report"]
naive = [
    (h.metadata or {}).get("product")
    for h in all_reports
    if re.search(r"\bsoccer\b", h.page_content, re.IGNORECASE)
]
print(f"NAIVE substring 'soccer' → {len(naive)} hits: {naive}")

# Semantic vector search via OracleVS.
hits = oracle_vs.similarity_search("soccer merchandise demand", k=5)
print(f"\nSEMANTIC 'soccer merchandise demand' → {len(hits)} hits:")
for d in hits:
    print(f"  {d.page_content.splitlines()[0]}")

In [ ]:
# Hard-stop checkpoint: TODO 6 — naive should be empty, semantic non-empty.
assert len(naive) == 0, (
    f"❌ TODO 6: naive substring matched {naive}. Did the seed step somehow "
    "include the literal word 'soccer'? Re-check seed_supplychain.py."
)
assert len(hits) >= 3, (
    f"❌ TODO 6: semantic search only returned {len(hits)} hits — expected ≥ 3."
)
print(f"✅ TODO 6 passed — keyword: {len(naive)}, semantic: {len(hits)}.")

> **Key takeaway — Part 7.** Real catalogues use brand names, product
> codes, abbreviations. A keyword search forces the planner to know the
> internal vocabulary; semantic search lets them ask in their own words.

# Part 8 — `demand_analyst` specialist

First specialist. One tool: semantic search over `OracleVS`. Its job is
to take a category or product line, retrieve comparable historical
reports, and return a tight 3–5-sentence summary.

📖 [`docs/part-8-demand-analyst.md`](../docs/part-8-demand-analyst.md)

## 8.1 Define the tool + compile the specialist

> **TODO 7.** Build a single-tool specialist that searches `OracleVS` for
> demand reports.
>
> **Steps**
> 1. Implement `search_demand_reports(query)` — call
>    `oracle_vs.similarity_search(query, k=5)` and join the page contents
>    with `"\n\n---\n\n"`.
> 2. `agent_model = ChatOpenAI(model=LLM_MODEL, **chat_model_kwargs())`.
> 3. Compile `demand_analyst` with `create_agent(...)`. Required kwargs:
>    - `tools=[search_demand_reports]`
>    - `system_prompt=...`
>    - `name="demand_analyst"`  ← the supervisor uses this name in handoffs
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-8-demand-analyst.md](../docs/part-8-demand-analyst.md)

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def search_demand_reports(query: str) -> str:
    """Search historical product demand reports by semantic similarity."""
    docs = oracle_vs.similarity_search(query, k=5)
    if not docs:
        return "No matches."
    return "\n\n---\n\n".join(d.page_content for d in docs)


agent_model = ChatOpenAI(model=LLM_MODEL, **chat_model_kwargs())

demand_analyst = create_agent(
    agent_model,
    tools=[search_demand_reports],
    system_prompt=(
        "You are a demand analyst. Given a category or product line, use "
        "`search_demand_reports` and return a concise 3-5 sentence summary "
        "covering volume direction, peak hours/days, and unique-visitor coverage."
    ),
    name="demand_analyst",
)
print(f"demand_analyst compiled on {LLM_PROVIDER}/{LLM_MODEL}")

In [ ]:
# Hard-stop checkpoint: TODO 7 — the specialist should compile and carry
# its `name` so the supervisor can hand off to it.
assert demand_analyst is not None, "❌ TODO 7: demand_analyst is still None."
assert getattr(demand_analyst, "name", None) == "demand_analyst", (
    "❌ TODO 7: missing `name=\"demand_analyst\"` in create_agent(...)."
)
print("✅ TODO 7 passed — demand_analyst is a compiled, named agent.")

> **Key takeaway — Part 8.** A specialist agent is `create_agent` + a
> focused prompt + a small toolbox. Naming it
> (`name="demand_analyst"`) matters: the supervisor uses that name in
> hand-off tool calls.

# Part 9 — `policy_agent` specialist

Second specialist. Two tools because it has two distinct lookups:

- **`get_planner_policy`** — pulls the standing buy-volume policy memo
  from `OracleVS`.
- **`get_user_memory(user_id)`** — pulls the active planner's saved
  preferences from `AsyncOracleStore`. The supervisor passes the
  `user_id` through.

📖 [`docs/part-9-policy-agent.md`](../docs/part-9-policy-agent.md)

## 9.1 Define both tools + compile the specialist

> **TODO 8.** Build a two-tool specialist:
> - `get_planner_policy()` — fetch the policy memo from `OracleVS`.
> - `get_user_memory(user_id)` — fetch the active planner's preferences
>   from `AsyncOracleStore`. Must be `async def`.
>
> **Hints**
> - `get_planner_policy`: `oracle_vs.similarity_search("planner buy volume policy", k=1)`.
> - `get_user_memory`:
>   ```python
>   items = await agent_store.asearch(
>       ("users", user_id, "memories"),
>       query="preference",
>       limit=3,
>   )
>   ```
>
> Then compile `policy_agent` with `create_agent(..., tools=[both], name="policy_agent")` and a
> prompt that tells it the `user_id` arrives inside the supervisor's request.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-9-policy-agent.md](../docs/part-9-policy-agent.md)

In [ ]:
@tool
def get_planner_policy() -> str:
    """Fetch the standing planner-prefs buy-volume policy from OracleVS."""
    docs = oracle_vs.similarity_search("planner buy volume policy", k=1)
    return docs[0].page_content if docs else "No policy on file."


@tool
async def get_user_memory(user_id: str) -> str:
    """Look up long-term saved preferences for a planner by their user_id."""
    items = await agent_store.asearch(
        ("users", user_id, "memories"),
        query="preference",
        limit=3,
    )
    if not items:
        return f"No saved memories for user_id={user_id}."
    return "\n".join(f"- {it.value.get('note', '')}" for it in items)


policy_agent = create_agent(
    agent_model,
    tools=[get_planner_policy, get_user_memory],
    system_prompt=(
        "You are the policy and preference agent. Use `get_planner_policy` for the "
        "standing buy-volume policy, and `get_user_memory(user_id=...)` for the active "
        "planner's saved preferences (the user_id is mentioned in the supervisor's "
        "request). Return both verbatim — do not editorialise."
    ),
    name="policy_agent",
)
print(f"policy_agent compiled on {LLM_PROVIDER}/{LLM_MODEL}")

In [ ]:
# Hard-stop checkpoint: TODO 8 — both tools must work; both records reachable.
assert policy_agent is not None, "❌ TODO 8: policy_agent is still None."
_policy = get_planner_policy.invoke({})
assert "policy" in _policy.lower() or "500" in _policy, (
    f"❌ TODO 8a: get_planner_policy returned an unexpected payload: {_policy[:120]}"
)
_priya = await get_user_memory.ainvoke({"user_id": "priya"})
assert "conservative" in _priya.lower(), (
    f"❌ TODO 8b: get_user_memory(\"priya\") didn't return her conservative pref: {_priya[:120]}"
)
print("✅ TODO 8 passed — both tools work; policy + Priya's memory retrievable.")

> **Key takeaway — Part 9.** Small, focused tool surfaces per specialist
> are the point. Each agent gets exactly the tools its job needs.

# Part 10 — Supervisor + end-to-end run

`langgraph_supervisor.create_supervisor(...)` builds an orchestrator
agent that exposes each specialist as a **handoff tool**. When the
supervisor decides *"I need policy + preferences,"* it emits a tool call
that hands control to the `policy_agent` sub-graph. The specialist runs
its own ReAct loop and returns a single summary. Control goes back to
the supervisor — which can call the other specialist, or synthesise the
final answer.

The supervisor is doing **planning and synthesis**; the specialists are
doing **focused retrieval**.

📖 [`docs/part-10-supervisor.md`](../docs/part-10-supervisor.md)

![Multi-agent topology — supervisor, specialists, and Oracle](../images/zoomed_in_multi_agent_overview.png)


## 10.1 Compile the supervisor and invoke it

> **TODO 9.** Tie both specialists together with `langgraph_supervisor`,
> then invoke and bind the answer.
>
> **Steps**
> 1. `supervisor_graph = create_supervisor([demand_analyst, policy_agent], model=agent_model, prompt=...)`.
> 2. Compile with **both** memory layers:
>    - `checkpointer=saver` (per-thread short-term)
>    - `store=agent_store` (cross-thread long-term)
> 3. `await supervisor.ainvoke({"messages": [...]}, config={"configurable": {"thread_id": THREAD_ID}})`.
> 4. Bind `final = result["messages"][-1].content` so the checkpoint can verify it.
>
> 📖 Full walkthrough + copy-pasteable solution: [docs/part-10-supervisor.md](../docs/part-10-supervisor.md)

In [ ]:
from langgraph_supervisor import create_supervisor

supervisor_graph = create_supervisor(
    [demand_analyst, policy_agent],
    model=agent_model,
    prompt=(
        "You are the supply-chain planning supervisor. For any planner request:\n"
        "1. Delegate to `policy_agent` for the standing policy + the active planner's "
        "user_id memories.\n"
        "2. Delegate to `demand_analyst` for historical demand data on the relevant categories.\n"
        "3. Synthesise a concise buy recommendation that respects BOTH the policy and the "
        "user's saved preferences. Cite the data inline."
    ),
)

supervisor = supervisor_graph.compile(
    checkpointer=saver,
    store=agent_store,
)
print(f"multi-agent supervisor compiled on {LLM_PROVIDER}/{LLM_MODEL}")

NEW_REQUEST = (
    "I'm planner with user_id=priya. We're debating how aggressively to stock "
    "soccer / football merchandise for the upcoming season. Pull demand intel "
    "from comparable SKUs in our history and draft a buy recommendation that "
    "respects my preferences and the standing policy."
)
THREAD_ID = "demand-plan-soccer-multiagent"

result = await supervisor.ainvoke(
    {"messages": [{"role": "user", "content": NEW_REQUEST}]},
    config={"configurable": {"thread_id": THREAD_ID}},
)
final = result["messages"][-1].content
print(final)

In [ ]:
# Hard-stop checkpoint: TODO 9 — the answer must reference BOTH:
#   - the policy threshold (500 unique IPs)  → policy_agent + OracleVS
#   - Priya's saved conservative preference  → policy_agent + AsyncOracleStore
assert supervisor is not None, "❌ TODO 9: supervisor is still None."
assert final, "❌ TODO 9: `final` is empty — did you invoke and bind the answer?"
_text = final.lower()
assert "500" in _text or "policy" in _text, (
    "❌ TODO 9: the answer doesn't reference the policy threshold. "
    "Did the supervisor call policy_agent?"
)
assert "conservative" in _text or "priya" in _text, (
    "❌ TODO 9: the answer doesn't reference Priya's saved preference. "
    "Did the supervisor pass user_id=priya through? Did you compile with store=agent_store?"
)
print("✅ TODO 9 passed — answer weaves both the policy (OracleVS) and the user's preference (AsyncOracleStore).")

> **Key takeaway — Part 10.** One `.compile(checkpointer=..., store=...)`
> wires both memory horizons into the entire multi-agent graph. The
> answer above weaves a fact from `OracleVS` (policy threshold), a fact
> from `AsyncOracleStore` (Priya's preference), and specific historical
> SKUs (also `OracleVS`) — no single tool call could have produced
> that.

# Part 11 — `OracleChatMessageHistory` — durable session transcripts

Not every LangChain app is a LangGraph state machine. Sometimes you just
want a plain chat history — a `session_id`, a list of messages, full
persistence, no graph. That's `OracleChatMessageHistory`.

📖 [`docs/part-11-chat-history.md`](../docs/part-11-chat-history.md)

## 11.1 A standalone session

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_oracledb.chat_message_histories import OracleChatMessageHistory

session_id = "planner-priya-2026Q3"
history = OracleChatMessageHistory(
    session_id=session_id,
    client=oracle_client,
    table_name="langchain_planner_chat",
)
history.clear()

history.add_messages([
    HumanMessage(content="What was our latest soccer merchandise buy recommendation?"),
    AIMessage(content="See the recommendation above — conservative volumes on cleats, evening promo support."),
    HumanMessage(content="Got it. And the hydration SKU push?"),
    AIMessage(content="No standing recommendation yet — fresh planner ran an exploratory query."),
])

print(f"session {session_id!r} now has {len(history.messages)} turns:")
for m in history.messages:
    print(f"  [{type(m).__name__}] {m.content}")

history2 = OracleChatMessageHistory(
    session_id=session_id,
    client=oracle_client,
    table_name="langchain_planner_chat",
)
print(f"\nreopened session {session_id!r}: {len(history2.messages)} turns recovered from Oracle.")

> **Key takeaway — Part 11.** Pick the right primitive. Use LangGraph's
> checkpointer + store when you have a multi-step agent; use
> `OracleChatMessageHistory` when you have a plain chatbot.

# Part 12 — Teardown

Close the three connections we opened in Part 1.

In [ ]:
oracle_client.close()
await saver_conn.close()
await store_conn.close()
print("connections closed")

## Done — what you just built

- **Single substrate, six primitives.** One Oracle, in-database
  embeddings, `OracleVS`, `AsyncOracleStore`, `AsyncOracleSaver`,
  `OracleSemanticCache`, `OracleChatMessageHistory`.
- **Multi-agent decomposition.** Two specialists with focused tool
  surfaces, a supervisor that decomposes + synthesises.
- **Dual-memory wiring.** `.compile(checkpointer=..., store=...)` is the
  single place where short-term per-thread memory and long-term
  cross-thread memory converge.

Open the chat app on **port 3000** to see the same supervisor + two
specialists running with a real-time UI.